# depgraph

> Hierarchical dependency graph with symbol-level tracking and duplication detection

In [ ]:
#| default_exp utils.depgraph

In [ ]:
#| hide
from nbdev.showdoc import *

## Imports

In [ ]:
#| export
from __future__ import annotations
from typing import Any, Dict, List, Optional, Set, Tuple, NamedTuple
from dataclasses import dataclass, field
from pathlib import Path
from collections import defaultdict
import ast
import hashlib
import json
import difflib

from nbdev_mcp.utils.paths import (
    resolve_selector, iter_notebooks, read_nb, settings_dict, nbs_dir
)
from nbdev_mcp.utils.nb import (
    join_source, find_default_exp, extract_symbols, extract_imports,
    Symbol, Import, Cell, NBFile
)

## Data Model

In [ ]:
#| export
@dataclass
class SymbolNode:
    """A symbol in the dependency graph.
    
    Parameters
    ----------
    id : str
        Unique identifier (module.notebook.symbol).
    name : str
        Symbol name.
    kind : str
        Type: 'function', 'class', 'constant', 'type'.
    notebook : str
        Source notebook path.
    module : str
        Target module name.
    cell_index : int
        Cell index in notebook.
    source : str
        Source code.
    source_hash : str
        Hash of normalized source for duplication detection.
    line_count : int
        Number of lines of code.
    """
    id: str
    name: str
    kind: str
    notebook: str
    module: str
    cell_index: int
    source: str = ''
    source_hash: str = ''
    line_count: int = 0
    
    def __post_init__(self):
        if self.source and not self.source_hash:
            # Normalize source for comparison
            normalized = self._normalize_source(self.source)
            self.source_hash = hashlib.md5(normalized.encode()).hexdigest()[:12]
            self.line_count = len(self.source.strip().splitlines())
    
    @staticmethod
    def _normalize_source(source: str) -> str:
        """Normalize source for comparison (remove comments, normalize whitespace)."""
        try:
            tree = ast.parse(source)
            return ast.dump(tree)
        except:
            # Fallback to simple normalization
            lines = [l.strip() for l in source.splitlines() if l.strip() and not l.strip().startswith('#')]
            return '\n'.join(lines)

In [ ]:
#| export
@dataclass
class NotebookNode:
    """A notebook in the dependency graph.
    
    Parameters
    ----------
    id : str
        Unique identifier (relative path).
    path : Path
        Full path to notebook.
    module : str
        Target module (from default_exp).
    symbols : List[SymbolNode]
        Symbols defined in this notebook.
    """
    id: str
    path: Path
    module: str
    symbols: List[SymbolNode] = field(default_factory=list)

In [ ]:
#| export
@dataclass
class ModuleNode:
    """A module (package) in the dependency graph.
    
    Parameters
    ----------
    id : str
        Module name.
    notebooks : List[NotebookNode]
        Notebooks that export to this module.
    """
    id: str
    notebooks: List[NotebookNode] = field(default_factory=list)

In [ ]:
#| export
@dataclass
class Edge:
    """A dependency edge in the graph.
    
    Parameters
    ----------
    source : str
        Source node ID (the supporter/dependency).
    target : str
        Target node ID (the dependent).
    kind : str
        Edge type: 'import', 'usage', 'duplicate_exact', 'duplicate_similar', 'duplicate_suspected'.
    weight : float
        Edge weight (1.0 for imports, similarity score for duplicates).
    metadata : Dict
        Additional edge metadata.
    """
    source: str
    target: str
    kind: str = 'import'
    weight: float = 1.0
    metadata: Dict[str, Any] = field(default_factory=dict)

In [ ]:
#| export
@dataclass
class DependencyGraph:
    """Complete hierarchical dependency graph.
    
    Parameters
    ----------
    project : Path
        Project root path.
    lib_name : str
        Library name.
    modules : Dict[str, ModuleNode]
        Module nodes by ID.
    notebooks : Dict[str, NotebookNode]
        Notebook nodes by ID.
    symbols : Dict[str, SymbolNode]
        Symbol nodes by ID.
    edges : List[Edge]
        All edges in the graph.
    """
    project: Path
    lib_name: str
    modules: Dict[str, ModuleNode] = field(default_factory=dict)
    notebooks: Dict[str, NotebookNode] = field(default_factory=dict)
    symbols: Dict[str, SymbolNode] = field(default_factory=dict)
    edges: List[Edge] = field(default_factory=list)
    
    def add_edge(self, source: str, target: str, kind: str = 'import', weight: float = 1.0, **metadata):
        """Add an edge to the graph."""
        self.edges.append(Edge(source, target, kind, weight, metadata))
    
    def get_node_edges(self, node_id: str, direction: str = 'both') -> List[Edge]:
        """Get edges for a node."""
        result = []
        for e in self.edges:
            if direction in ('both', 'out') and e.source == node_id:
                result.append(e)
            if direction in ('both', 'in') and e.target == node_id:
                result.append(e)
        return result
    
    def to_dict(self) -> Dict[str, Any]:
        """Convert to dictionary for JSON serialization."""
        return {
            'project': str(self.project),
            'lib_name': self.lib_name,
            'modules': {
                k: {'id': v.id, 'notebooks': [n.id for n in v.notebooks]}
                for k, v in self.modules.items()
            },
            'notebooks': {
                k: {'id': v.id, 'path': str(v.path), 'module': v.module, 'symbols': [s.id for s in v.symbols]}
                for k, v in self.notebooks.items()
            },
            'symbols': {
                k: {'id': v.id, 'name': v.name, 'kind': v.kind, 'notebook': v.notebook,
                    'module': v.module, 'cell_index': v.cell_index, 'source_hash': v.source_hash,
                    'line_count': v.line_count}
                for k, v in self.symbols.items()
            },
            'edges': [
                {'source': e.source, 'target': e.target, 'kind': e.kind, 'weight': e.weight, 'metadata': e.metadata}
                for e in self.edges
            ]
        }

## Graph Building

In [ ]:
#| export
def extract_symbol_source(source: str, symbol: Symbol) -> str:
    """Extract the source code for a specific symbol from cell source.
    
    Parameters
    ----------
    source : str
        Full cell source code.
    symbol : Symbol
        Symbol to extract.
    
    Returns
    -------
    str
        Source code for just this symbol.
    """
    try:
        tree = ast.parse(source)
        for node in ast.walk(tree):
            if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)) and node.name == symbol.name:
                return ast.get_source_segment(source, node) or ''
            if isinstance(node, ast.ClassDef) and node.name == symbol.name:
                return ast.get_source_segment(source, node) or ''
            if isinstance(node, ast.Assign):
                for target in node.targets:
                    if isinstance(target, ast.Name) and target.id == symbol.name:
                        return ast.get_source_segment(source, node) or ''
    except:
        pass
    return ''

In [ ]:
#| export
def build_graph(project: Optional[str] = None) -> DependencyGraph:
    """Build a complete dependency graph for a project.
    
    Parameters
    ----------
    project : str, optional
        Project path or alias.
    
    Returns
    -------
    DependencyGraph
        Complete hierarchical dependency graph.
    """
    p = resolve_selector(project)
    settings = settings_dict(p)
    lib_name = settings.get('lib_name', 'pkg')
    # Handle hyphen vs underscore in lib names
    lib_name_underscore = lib_name.replace('-', '_')
    nbs = nbs_dir(p)
    
    graph = DependencyGraph(project=p, lib_name=lib_name)
    
    # Track symbol definitions for cross-referencing
    symbol_defs: Dict[str, str] = {}  # symbol_name -> symbol_id
    
    # First pass: collect all symbols
    for nb_path in iter_notebooks(p):
        nb_data = read_nb(nb_path)
        module = find_default_exp(nb_data) or ''
        nb_id = str(nb_path.relative_to(p))
        
        # Create notebook node
        nb_node = NotebookNode(id=nb_id, path=nb_path, module=module)
        graph.notebooks[nb_id] = nb_node
        
        # Ensure module node exists
        mod_parts = module.split('.') if module else ['_root']
        mod_id = mod_parts[0]
        if mod_id not in graph.modules:
            graph.modules[mod_id] = ModuleNode(id=mod_id)
        graph.modules[mod_id].notebooks.append(nb_node)
        
        # Extract symbols from each code cell
        for i, cell in enumerate(nb_data.get('cells', [])):
            if cell.get('cell_type') != 'code':
                continue
            src = join_source(cell.get('source', []))
            
            # Check if this is an export cell
            if not any(d in src for d in ['#| export', '#|export']):
                continue
            
            symbols = extract_symbols(src)
            for sym in symbols:
                sym_id = f"{module}.{sym.name}" if module else sym.name
                sym_source = extract_symbol_source(src, sym)
                
                sym_node = SymbolNode(
                    id=sym_id,
                    name=sym.name,
                    kind=sym.kind,
                    notebook=nb_id,
                    module=module,
                    cell_index=i,
                    source=sym_source
                )
                graph.symbols[sym_id] = sym_node
                nb_node.symbols.append(sym_node)
                symbol_defs[sym.name] = sym_id
    
    # Second pass: build dependency edges from imports
    for nb_id, nb_node in graph.notebooks.items():
        nb_data = read_nb(nb_node.path)
        
        for i, cell in enumerate(nb_data.get('cells', [])):
            if cell.get('cell_type') != 'code':
                continue
            src = join_source(cell.get('source', []))
            
            imports = extract_imports(src)
            for imp in imports:
                # Check if this is an internal import (handle both hyphen and underscore)
                if imp.module.startswith(lib_name_underscore):
                    # Create edges for each imported name
                    for name in imp.names:
                        # Find the source symbol
                        source_mod = imp.module.replace(lib_name_underscore + '.', '')
                        source_id = f"{source_mod}.{name}"
                        
                        # Check if source symbol exists
                        if source_id not in graph.symbols:
                            continue
                        
                        # Create edge from source to notebook (notebook depends on source)
                        # Find symbols in this notebook that use the import
                        for sym in nb_node.symbols:
                            if sym.cell_index >= i and name in sym.source:
                                graph.add_edge(source_id, sym.id, 'import',
                                               module=imp.module, imported_name=name)
                                break
                        else:
                            # No specific symbol found, create notebook-level edge
                            graph.add_edge(source_id, nb_id, 'import',
                                           module=imp.module, imported_name=name)
    
    return graph

## Duplication Detection

In [ ]:
#| export
def detect_duplicates(
    graph: DependencyGraph,
    similarity_threshold: float = 0.8
) -> List[Edge]:
    """Detect potential code duplications in the graph.
    
    Parameters
    ----------
    graph : DependencyGraph
        The dependency graph to analyze.
    similarity_threshold : float
        Minimum similarity ratio for 'similar' duplicates (0-1).
    
    Returns
    -------
    List[Edge]
        Duplication edges (exact, similar, suspected).
    """
    dup_edges = []
    symbols = list(graph.symbols.values())
    
    # Group by hash for exact duplicates
    hash_groups: Dict[str, List[SymbolNode]] = defaultdict(list)
    for sym in symbols:
        if sym.source_hash and sym.line_count >= 3:  # Only check non-trivial code
            hash_groups[sym.source_hash].append(sym)
    
    # Create edges for exact duplicates
    for hash_val, group in hash_groups.items():
        if len(group) > 1:
            for i, sym1 in enumerate(group):
                for sym2 in group[i+1:]:
                    dup_edges.append(Edge(
                        source=sym1.id,
                        target=sym2.id,
                        kind='duplicate_exact',
                        weight=1.0,
                        metadata={'hash': hash_val, 'lines': sym1.line_count}
                    ))
    
    # Check for similar duplicates (same kind, similar structure)
    for i, sym1 in enumerate(symbols):
        if sym1.line_count < 5:  # Skip very short code
            continue
        for sym2 in symbols[i+1:]:
            if sym2.line_count < 5:
                continue
            if sym1.kind != sym2.kind:
                continue
            if sym1.source_hash == sym2.source_hash:
                continue  # Already handled as exact
            
            # Compare normalized sources
            ratio = difflib.SequenceMatcher(
                None,
                sym1._normalize_source(sym1.source),
                sym2._normalize_source(sym2.source)
            ).ratio()
            
            if ratio >= similarity_threshold:
                dup_edges.append(Edge(
                    source=sym1.id,
                    target=sym2.id,
                    kind='duplicate_similar',
                    weight=ratio,
                    metadata={'similarity': round(ratio, 3)}
                ))
            elif ratio >= 0.5:  # Suspected duplicates
                dup_edges.append(Edge(
                    source=sym1.id,
                    target=sym2.id,
                    kind='duplicate_suspected',
                    weight=ratio,
                    metadata={'similarity': round(ratio, 3)}
                ))
    
    return dup_edges

## Interactive Visualization

In [ ]:
#| export

D3_PACK_HTML_TEMPLATE = '''
<!DOCTYPE html>
<html>
<head>
    <meta charset="utf-8">
    <title>Dependency Graph - __LIB_NAME__</title>
    <script src="https://d3js.org/d3.v7.min.js"></script>
    <style>
        * { box-sizing: border-box; }
        body { margin: 0; font-family: -apple-system, BlinkMacSystemFont, sans-serif; background: #0d1117; color: #c9d1d9; }
        #container { display: flex; height: 100vh; }
        #graph { flex: 1; overflow: hidden; }
        #sidebar { width: 280px; padding: 12px; background: #161b22; overflow-y: auto; font-size: 12px; border-left: 1px solid #30363d; }
        h3 { margin: 0 0 8px 0; color: #58a6ff; font-size: 12px; text-transform: uppercase; letter-spacing: 0.5px; }
        .control { margin: 5px 0; }
        .control label { display: flex; align-items: center; cursor: pointer; }
        .control input { margin-right: 6px; }
        .legend { margin-top: 12px; padding-top: 12px; border-top: 1px solid #30363d; }
        .legend-item { display: flex; align-items: center; margin: 4px 0; font-size: 11px; }
        .legend-color { width: 12px; height: 12px; border-radius: 50%; margin-right: 6px; }
        .legend-line { width: 18px; height: 2px; margin-right: 6px; }
        #info { margin-top: 12px; padding: 10px; background: #21262d; border-radius: 6px; font-size: 11px; border: 1px solid #30363d; }
        #info h4 { margin: 0 0 6px 0; color: #f0883e; font-size: 13px; }
        #info p { margin: 4px 0; }
        svg { width: 100%; height: 100%; background: #0d1117; }
        .module-container { cursor: move; }
        .module-container > rect { fill: rgba(56, 139, 253, 0.06); stroke: rgba(56, 139, 253, 0.4); stroke-width: 2; transition: width 0.15s, height 0.15s; }
        .module-container > text { fill: #58a6ff; font-size: 14px; font-weight: 600; pointer-events: none; }
        .notebook-container { cursor: move; }
        .notebook-container > rect { fill: rgba(63, 185, 80, 0.06); stroke: rgba(63, 185, 80, 0.35); stroke-width: 1.5; transition: width 0.15s, height 0.15s; }
        .notebook-container > text { fill: #3fb950; font-size: 11px; font-weight: 500; pointer-events: none; }
        .symbol { cursor: grab; }
        .symbol:active { cursor: grabbing; }
        .symbol circle { stroke-width: 2; }
        .symbol text { fill: #c9d1d9; pointer-events: none; font-size: 9px; }
        .symbol.highlighted circle { stroke: #fff; stroke-width: 3; }
        .symbol.dimmed { opacity: 0.15; }
        .edge { fill: none; stroke-opacity: 0.5; }
        .edge.import { stroke: #4ecdc4; stroke-width: 1.5; }
        .edge.duplicate_exact { stroke: #f85149; stroke-width: 2.5; }
        .edge.duplicate_similar { stroke: #d29922; stroke-width: 2; stroke-dasharray: 5,3; }
        .edge.duplicate_suspected { stroke: #8b8b00; stroke-width: 1.5; stroke-dasharray: 3,3; }
        .edge.highlighted { stroke-opacity: 1; stroke-width: 3; }
        .edge.dimmed { stroke-opacity: 0.08; }
        button { background: #238636; color: white; border: none; padding: 6px 10px; border-radius: 6px; cursor: pointer; width: 100%; margin-top: 8px; font-size: 11px; }
        button:hover { background: #2ea043; }
        #search { width: 100%; padding: 8px; border: 1px solid #30363d; border-radius: 6px; background: #0d1117; color: #c9d1d9; margin-bottom: 12px; font-size: 12px; }
        #search:focus { outline: none; border-color: #58a6ff; }
    </style>
</head>
<body>
    <div id="container">
        <div id="graph"></div>
        <div id="sidebar">
            <input type="text" id="search" placeholder="Search symbols...">
            
            <h3>Edges</h3>
            <div class="control"><label><input type="checkbox" id="showImports" checked> Imports</label></div>
            <div class="control"><label><input type="checkbox" id="showDupExact"> Exact duplicates</label></div>
            <div class="control"><label><input type="checkbox" id="showDupSimilar"> Similar code</label></div>
            <div class="control"><label><input type="checkbox" id="showDupSuspected"> Suspected</label></div>
            
            <h3 style="margin-top: 12px;">Display</h3>
            <div class="control"><label><input type="checkbox" id="showLabels" checked> Symbol labels</label></div>
            <div class="control"><label><input type="checkbox" id="showModules" checked> Module boxes</label></div>
            <div class="control"><label><input type="checkbox" id="showNotebooks" checked> Notebook boxes</label></div>
            <button id="autoLayout">Auto Layout</button>
            <button id="fitContainers">Fit Containers</button>
            
            <div class="legend">
                <h3>Symbols</h3>
                <div class="legend-item"><div class="legend-color" style="background: #f0883e;"></div> Function</div>
                <div class="legend-item"><div class="legend-color" style="background: #a371f7;"></div> Class</div>
                <div class="legend-item"><div class="legend-color" style="background: #39d353;"></div> Constant</div>
                <h3 style="margin-top: 8px;">Edges</h3>
                <div class="legend-item"><div class="legend-line" style="background: #4ecdc4;"></div> Import</div>
                <div class="legend-item"><div class="legend-line" style="background: #f85149;"></div> Exact dup</div>
                <div class="legend-item"><div class="legend-line" style="background: #d29922;"></div> Similar</div>
            </div>
            
            <div id="info">
                <h4>Dependency Graph</h4>
                <p>Drag to move. Containers auto-resize to fit contents.</p>
            </div>
        </div>
    </div>
    <script>
const data = __GRAPH_JSON__;
const width = document.getElementById("graph").clientWidth;
const height = window.innerHeight;

const kindColor = d3.scaleOrdinal()
    .domain(["function", "class", "constant", "type"])
    .range(["#f0883e", "#a371f7", "#39d353", "#79c0ff"]);

// Build hierarchy
const modules = {};
const notebooks = {};
const symbols = {};

Object.entries(data.modules).forEach(([modId, mod]) => {
    modules[modId] = { id: modId, name: modId, notebooks: [], x: 0, y: 0, width: 200, height: 150 };
});

Object.entries(data.notebooks).forEach(([nbId, nbData]) => {
    const modId = nbData.module.split(".")[0] || "_root";
    if (!modules[modId]) modules[modId] = { id: modId, name: modId, notebooks: [], x: 0, y: 0, width: 200, height: 150 };
    
    const nbName = nbId.split("/").pop().replace(".ipynb", "");
    const nb = { id: nbId, name: nbName, moduleId: modId, module: modules[modId], symbols: [], localX: 0, localY: 0, width: 120, height: 80 };
    notebooks[nbId] = nb;
    modules[modId].notebooks.push(nb);
});

Object.entries(data.symbols).forEach(([symId, sym]) => {
    const nb = notebooks[sym.notebook];
    if (!nb) return;
    const symObj = {
        id: symId, name: sym.name, kind: sym.kind, notebookId: sym.notebook, notebook: nb, module: nb.module,
        lines: sym.line_count || 1, r: Math.max(5, Math.min(14, Math.sqrt(sym.line_count || 1) * 2.5)), localX: 0, localY: 0
    };
    symbols[symId] = symObj;
    nb.symbols.push(symObj);
});

const edges = data.edges.map(e => ({
    source: symbols[e.source], target: symbols[e.target], kind: e.kind, weight: e.weight || 1
})).filter(e => e.source && e.target);

// Spacing constants
const LABEL_HEIGHT = 14;  // Space for label below symbol
const SYM_SPACING_X = 55; // Horizontal space between symbols
const SYM_SPACING_Y = 48; // Vertical space (includes label)
const NB_PADDING_X = 18;  // Notebook left padding
const NB_PADDING_Y = 24;  // Notebook top padding (for title)
const NB_SPACING_X = 20;  // Space between notebooks
const NB_SPACING_Y = 20;  // Space between notebook rows
const MOD_PADDING_X = 15; // Module left padding
const MOD_PADDING_Y = 28; // Module top padding (for title)
const MOD_SPACING_X = 30; // Space between modules
const MOD_SPACING_Y = 30; // Space between module rows

// Size computation
function computeNotebookSize(nb) {
    if (nb.symbols.length === 0) return { width: 80, height: 50 };
    let maxX = 0, maxY = 0;
    nb.symbols.forEach(s => {
        maxX = Math.max(maxX, s.localX + s.r + 25);
        maxY = Math.max(maxY, s.localY + s.r + LABEL_HEIGHT + 8);
    });
    return { width: Math.max(80, maxX), height: Math.max(50, maxY) };
}

function computeModuleSize(mod) {
    if (mod.notebooks.length === 0) return { width: 120, height: 80 };
    let maxX = 0, maxY = 0;
    mod.notebooks.forEach(nb => {
        maxX = Math.max(maxX, nb.localX + nb.width + 10);
        maxY = Math.max(maxY, nb.localY + nb.height + 10);
    });
    return { width: Math.max(120, maxX), height: Math.max(80, maxY) };
}

function updateAllSizes() {
    Object.values(notebooks).forEach(nb => {
        const size = computeNotebookSize(nb);
        nb.width = size.width; nb.height = size.height;
    });
    Object.values(modules).forEach(mod => {
        const size = computeModuleSize(mod);
        mod.width = size.width; mod.height = size.height;
    });
}

function updateContainerRects() {
    moduleLayer.selectAll(".module-container").select("rect.mod-bg")
        .attr("width", d => d.width).attr("height", d => d.height);
    moduleLayer.selectAll(".notebook-container").select("rect.nb-bg")
        .attr("width", d => d.width).attr("height", d => d.height);
}

// Layout symbols in grid within notebook
function layoutSymbols(nb) {
    const count = nb.symbols.length;
    if (count === 0) return;
    const cols = Math.max(1, Math.ceil(Math.sqrt(count * 1.2))); // Slightly wider than tall
    nb.symbols.forEach((sym, k) => {
        const col = k % cols, row = Math.floor(k / cols);
        sym.localX = NB_PADDING_X + col * SYM_SPACING_X;
        sym.localY = NB_PADDING_Y + row * SYM_SPACING_Y;
    });
}

// Layout notebooks in grid within module
function layoutNotebooks(mod) {
    const count = mod.notebooks.length;
    if (count === 0) return;
    
    // First layout symbols to get notebook sizes
    mod.notebooks.forEach(layoutSymbols);
    mod.notebooks.forEach(nb => {
        const size = computeNotebookSize(nb);
        nb.width = size.width; nb.height = size.height;
    });
    
    // Grid layout for notebooks
    const cols = Math.max(1, Math.ceil(Math.sqrt(count)));
    let rowHeights = [], colWidths = [];
    
    // Compute row heights and column widths
    mod.notebooks.forEach((nb, j) => {
        const col = j % cols, row = Math.floor(j / cols);
        rowHeights[row] = Math.max(rowHeights[row] || 0, nb.height);
        colWidths[col] = Math.max(colWidths[col] || 0, nb.width);
    });
    
    // Position notebooks
    mod.notebooks.forEach((nb, j) => {
        const col = j % cols, row = Math.floor(j / cols);
        let x = MOD_PADDING_X, y = MOD_PADDING_Y;
        for (let c = 0; c < col; c++) x += colWidths[c] + NB_SPACING_X;
        for (let r = 0; r < row; r++) y += rowHeights[r] + NB_SPACING_Y;
        nb.localX = x;
        nb.localY = y;
    });
}

// Initial layout
const moduleList = Object.values(modules);

// Layout all modules
moduleList.forEach(layoutNotebooks);
updateAllSizes();

// Position modules in grid
const modCols = Math.max(1, Math.ceil(Math.sqrt(moduleList.length)));
let modRowHeights = [], modColWidths = [];

moduleList.forEach((mod, i) => {
    const col = i % modCols, row = Math.floor(i / modCols);
    modRowHeights[row] = Math.max(modRowHeights[row] || 0, mod.height);
    modColWidths[col] = Math.max(modColWidths[col] || 0, mod.width);
});

moduleList.forEach((mod, i) => {
    const col = i % modCols, row = Math.floor(i / modCols);
    let x = 40, y = 40;
    for (let c = 0; c < col; c++) x += modColWidths[c] + MOD_SPACING_X;
    for (let r = 0; r < row; r++) y += modRowHeights[r] + MOD_SPACING_Y;
    mod.x = x;
    mod.y = y;
});

function getSymbolPos(sym) { 
    const nb = sym.notebook, mod = nb.module;
    return { x: mod.x + nb.localX + sym.localX, y: mod.y + nb.localY + sym.localY }; 
}

const svg = d3.select("#graph").append("svg")
    .attr("viewBox", [0, 0, width, height])
    .call(d3.zoom().scaleExtent([0.1, 4]).on("zoom", ({transform}) => container.attr("transform", transform)));

const container = svg.append("g");
const edgeLayer = container.append("g").attr("class", "edge-layer");
const moduleLayer = container.append("g").attr("class", "module-layer");

svg.append("defs").append("marker")
    .attr("id", "arrow").attr("viewBox", "0 -4 8 8").attr("refX", 8).attr("refY", 0)
    .attr("markerWidth", 5).attr("markerHeight", 5).attr("orient", "auto")
    .append("path").attr("fill", "#4ecdc4").attr("d", "M0,-4L8,0L0,4Z");

function render() {
    const modGroups = moduleLayer.selectAll(".module-container")
        .data(moduleList, d => d.id).join("g")
        .attr("class", "module-container")
        .attr("transform", d => `translate(${d.x},${d.y})`)
        .call(d3.drag().on("drag", function(event, d) {
            d.x += event.dx; d.y += event.dy;
            d3.select(this).attr("transform", `translate(${d.x},${d.y})`);
            updateEdges();
        }));
    
    modGroups.selectAll("rect.mod-bg").data(d => [d]).join("rect")
        .attr("class", "mod-bg").attr("width", d => d.width).attr("height", d => d.height).attr("rx", 10);
    
    modGroups.selectAll("text.mod-label").data(d => [d]).join("text")
        .attr("class", "mod-label").attr("x", 12).attr("y", 20).text(d => d.name);
    
    const nbGroups = modGroups.selectAll(".notebook-container")
        .data(d => d.notebooks, d => d.id).join("g")
        .attr("class", "notebook-container")
        .attr("transform", d => `translate(${d.localX},${d.localY})`)
        .call(d3.drag().on("drag", function(event, d) {
            d.localX = Math.max(5, d.localX + event.dx);
            d.localY = Math.max(25, d.localY + event.dy);
            d3.select(this).attr("transform", `translate(${d.localX},${d.localY})`);
            updateAllSizes(); updateContainerRects(); updateEdges();
        }));
    
    nbGroups.selectAll("rect.nb-bg").data(d => [d]).join("rect")
        .attr("class", "nb-bg").attr("width", d => d.width).attr("height", d => d.height).attr("rx", 6);
    
    nbGroups.selectAll("text.nb-label").data(d => [d]).join("text")
        .attr("class", "nb-label").attr("x", 6).attr("y", 14)
        .text(d => d.name.length > 20 ? d.name.slice(0,18) + ".." : d.name);
    
    const symGroups = nbGroups.selectAll(".symbol")
        .data(d => d.symbols, d => d.id).join("g")
        .attr("class", "symbol")
        .attr("transform", d => `translate(${d.localX},${d.localY})`)
        .call(d3.drag().on("drag", function(event, d) {
            d.localX = Math.max(d.r + 5, d.localX + event.dx);
            d.localY = Math.max(d.r + 18, d.localY + event.dy);
            d3.select(this).attr("transform", `translate(${d.localX},${d.localY})`);
            updateAllSizes(); updateContainerRects(); updateEdges();
        }))
        .on("mouseover", (e, d) => highlight(d, true))
        .on("mouseout", (e, d) => { if (!locked) highlight(d, false); })
        .on("click", (e, d) => { e.stopPropagation(); toggleLock(d); });
    
    symGroups.selectAll("circle").data(d => [d]).join("circle")
        .attr("r", d => d.r).attr("fill", d => kindColor(d.kind))
        .attr("stroke", d => d3.color(kindColor(d.kind)).darker(0.6));
    
    symGroups.selectAll("text").data(d => [d]).join("text")
        .attr("dy", d => d.r + 10).attr("text-anchor", "middle")
        .text(d => d.name.length > 10 ? d.name.slice(0,8) + ".." : d.name);
    
    updateEdges();
}

function updateEdges() {
    const show = {
        import: document.getElementById("showImports").checked,
        duplicate_exact: document.getElementById("showDupExact").checked,
        duplicate_similar: document.getElementById("showDupSimilar").checked,
        duplicate_suspected: document.getElementById("showDupSuspected").checked
    };
    
    edgeLayer.selectAll("path")
        .data(edges.filter(e => show[e.kind]), d => d.source.id + "-" + d.target.id + "-" + d.kind)
        .join("path")
        .attr("class", d => "edge " + d.kind)
        .attr("marker-end", d => d.kind === "import" ? "url(#arrow)" : null)
        .attr("d", d => {
            const s = getSymbolPos(d.source), t = getSymbolPos(d.target);
            const dx = t.x - s.x, dy = t.y - s.y, dist = Math.sqrt(dx*dx + dy*dy);
            if (dist < 1) return "";
            const sr = d.source.r + 2, tr = d.target.r + 6;
            return `M${s.x + dx/dist*sr},${s.y + dy/dist*sr}L${t.x - dx/dist*tr},${t.y - dy/dist*tr}`;
        });
}

let locked = null;

function toggleLock(d) {
    if (locked === d.id) { locked = null; highlight(d, false); }
    else { if (locked) highlight(symbols[locked], false); locked = d.id; highlight(d, true); }
}

function highlight(d, on) {
    const connected = new Set();
    if (on) {
        edges.forEach(e => {
            if (e.source.id === d.id) connected.add(e.target.id);
            if (e.target.id === d.id) connected.add(e.source.id);
        });
        connected.add(d.id);
    }
    
    moduleLayer.selectAll(".symbol")
        .classed("highlighted", s => on && s.id === d.id)
        .classed("dimmed", s => on && !connected.has(s.id));
    
    edgeLayer.selectAll("path")
        .classed("highlighted", e => on && (e.source.id === d.id || e.target.id === d.id))
        .classed("dimmed", e => on && e.source.id !== d.id && e.target.id !== d.id);
    
    if (on) showInfo(d);
}

function showInfo(d) {
    const info = document.getElementById("info");
    const inE = edges.filter(e => e.target.id === d.id);
    const outE = edges.filter(e => e.source.id === d.id);
    let html = `<h4>${d.name}</h4><p><b>Type:</b> ${d.kind} | <b>Lines:</b> ${d.lines}</p><p><b>Notebook:</b> ${d.notebook.name}</p>`;
    if (outE.length) html += `<p><b>Supports (${outE.length}):</b> ${outE.slice(0,5).map(e=>e.target.name).join(", ")}${outE.length>5?"...":""}</p>`;
    if (inE.length) html += `<p><b>Uses (${inE.length}):</b> ${inE.slice(0,5).map(e=>e.source.name).join(", ")}${inE.length>5?"...":""}</p>`;
    info.innerHTML = html;
}

// Controls
document.getElementById("showImports").onchange = updateEdges;
document.getElementById("showDupExact").onchange = updateEdges;
document.getElementById("showDupSimilar").onchange = updateEdges;
document.getElementById("showDupSuspected").onchange = updateEdges;
document.getElementById("showLabels").onchange = function() { moduleLayer.selectAll(".symbol text").attr("opacity", this.checked ? 1 : 0); };
document.getElementById("showModules").onchange = function() { moduleLayer.selectAll(".module-container > rect, .module-container > text.mod-label").attr("opacity", this.checked ? 1 : 0); };
document.getElementById("showNotebooks").onchange = function() { moduleLayer.selectAll(".notebook-container > rect, .notebook-container > text.nb-label").attr("opacity", this.checked ? 1 : 0); };

document.getElementById("autoLayout").onclick = function() {
    moduleList.forEach(layoutNotebooks);
    updateAllSizes();
    
    // Recompute module grid
    let rh = [], cw = [];
    moduleList.forEach((mod, i) => {
        const col = i % modCols, row = Math.floor(i / modCols);
        rh[row] = Math.max(rh[row] || 0, mod.height);
        cw[col] = Math.max(cw[col] || 0, mod.width);
    });
    moduleList.forEach((mod, i) => {
        const col = i % modCols, row = Math.floor(i / modCols);
        let x = 40, y = 40;
        for (let c = 0; c < col; c++) x += cw[c] + MOD_SPACING_X;
        for (let r = 0; r < row; r++) y += rh[r] + MOD_SPACING_Y;
        mod.x = x; mod.y = y;
    });
    render();
};

document.getElementById("fitContainers").onclick = function() {
    updateAllSizes(); updateContainerRects();
};

document.getElementById("search").oninput = function() {
    const q = this.value.toLowerCase();
    moduleLayer.selectAll(".symbol").classed("dimmed", d => q && !d.name.toLowerCase().includes(q));
};

svg.on("click", () => { if (locked) { highlight(symbols[locked], false); locked = null; }});

render();
    </script>
</body>
</html>
'''

In [ ]:
#| export
def generate_interactive_graph(
    project: Optional[str] = None,
    output_path: Optional[str] = None,
    include_duplicates: bool = True,
    similarity_threshold: float = 0.8
) -> Dict[str, Any]:
    """Generate an interactive dependency graph visualization.
    
    Uses D3 circle packing to show hierarchical structure:
    - Modules contain notebooks contain symbols
    - Click to zoom into any level
    - Edges show import dependencies and code duplications
    
    Parameters
    ----------
    project : str, optional
        Project path or alias.
    output_path : str, optional
        Output HTML file path. Defaults to project/dep_graph.html.
    include_duplicates : bool
        Whether to detect and show code duplications.
    similarity_threshold : float
        Threshold for similar duplicate detection.
    
    Returns
    -------
    Dict[str, Any]
        Result with graph stats and output path.
    """
    p = resolve_selector(project)
    
    # Build the graph
    graph = build_graph(str(p))
    
    # Detect duplicates if requested
    if include_duplicates:
        dup_edges = detect_duplicates(graph, similarity_threshold)
        graph.edges.extend(dup_edges)
    
    # Generate HTML using string replacement (not .format() to avoid brace issues)
    graph_dict = graph.to_dict()
    html = D3_PACK_HTML_TEMPLATE.replace('__LIB_NAME__', graph.lib_name)
    html = html.replace('__GRAPH_JSON__', json.dumps(graph_dict))
    
    # Write output
    out_path = Path(output_path) if output_path else p / 'dep_graph.html'
    out_path.write_text(html, encoding='utf-8')
    
    # Count duplicates by type
    dup_counts = defaultdict(int)
    for e in graph.edges:
        if e.kind.startswith('duplicate'):
            dup_counts[e.kind] += 1
    
    return {
        'ok': True,
        'output_path': str(out_path),
        'stats': {
            'modules': len(graph.modules),
            'notebooks': len(graph.notebooks),
            'symbols': len(graph.symbols),
            'import_edges': sum(1 for e in graph.edges if e.kind == 'import'),
            'duplicates': dict(dup_counts)
        }
    }

## Next

In [ ]:
#| export



## Export

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()